Why Naive Bayes for Text Classification?

Based on Bayes’ theorem.
Assumes features (words) are conditionally independent given the class.
Works very well for spam filtering, sentiment analysis, topic classification.
Extremely fast even on large datasets.

In [1]:
# Step 1: Import libraries
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Step 2: Example dataset
texts = [
    "free money offer",          # spam
    "win lottery now",           # spam
    "earn money easily",         # spam
    "let's go for dinner",       # ham
    "I will call you tomorrow",  # ham
    "how about a game tonight"   # ham
]
labels = ["spam", "spam", "spam", "ham", "ham", "ham"]

# Step 3: Convert text to bag-of-words
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

# Step 4: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.3, random_state=42)

# Step 5: Train Naive Bayes classifier
model = MultinomialNB()
model.fit(X_train, y_train)

# Step 6: Predict
y_pred = model.predict(X_test)

print("Predictions:", y_pred)
print("Accuracy:", accuracy_score(y_test, y_pred))


Predictions: ['spam' 'ham']
Accuracy: 0.5


1. Binary Naive Bayes Classification

Naive Bayes works very well when you have two classes:
Spam vs Ham
Positive vs Negative
Fake vs Real

The formula is the same as before, but only two classes exist, so the model chooses whichever has the higher posterior probability.

🔹 2. Sentiment Analysis with Naive Bayes
We’ll classify text into positive or negative sentiments.

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Step 1: Example dataset (small for demo)
texts = [
    "I love this movie, it was fantastic!",   # positive
    "This film was awful, I hated it.",       # negative
    "What a wonderful experience!",           # positive
    "Terrible acting and boring story.",      # negative
    "Absolutely amazing performance!",        # positive
    "Worst movie I have ever seen."           # negative
]
labels = ["positive", "negative", "positive", "negative", "positive", "negative"]

# Step 2: Convert text → features
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

# Step 3: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.3, random_state=42)

# Step 4: Train Naive Bayes
model = MultinomialNB()
model.fit(X_train, y_train)

# Step 5: Predict
y_pred = model.predict(X_test)

print("Predictions:", y_pred)
print("Accuracy:", accuracy_score(y_test, y_pred))

# from sklearn.feature_extraction.text import TfidfVectorizer
# vectorizer = TfidfVectorizer()
# X = vectorizer.fit_transform(texts)


Predictions: ['positive' 'positive']
Accuracy: 0.5


Generative Model for Multinomial Naïve Bayes

In [4]:
import numpy as np
from collections import defaultdict, Counter

class MultinomialNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, docs, labels):
        self.classes = set(labels)
        self.priors = {}
        self.word_counts = {c: Counter() for c in self.classes}
        self.class_totals = {c: 0 for c in self.classes}
        
        # Count words per class
        for doc, label in zip(docs, labels):
            self.priors[label] = self.priors.get(label, 0) + 1
            for word in doc.split():
                self.word_counts[label][word] += 1
                self.class_totals[label] += 1
        
        # Normalize priors
        total_docs = len(labels)
        for c in self.classes:
            self.priors[c] /= total_docs

    def predict(self, doc):
        scores = {}
        words = doc.split()
        vocab = set(word for counts in self.word_counts.values() for word in counts)
        V = len(vocab)
        
        for c in self.classes:
            # log prior
            score = np.log(self.priors[c])
            # log likelihood
            for word in words:
                count_wc = self.word_counts[c][word]
                prob_wc = (count_wc + self.alpha) / (self.class_totals[c] + self.alpha * V)
                score += np.log(prob_wc)
            scores[c] = score
        return max(scores, key=scores.get)

# Example
docs = ["free money offer", "win lottery now", "call me tonight", "let's play football"]
labels = ["spam", "spam", "ham", "ham"]

model = MultinomialNB()
model.fit(docs, labels)

print(model.predict("free lottery money"))   # spam
print(model.predict("let's go tonight"))     # ham


spam
ham


#Precision, recall and F1

Suppose we classify emails as Spam (positive) or Ham (negative):

TP = 70 (spam correctly detected)

FP = 30 (ham misclassified as spam)

FN = 10 (spam missed as ham)

TN = 90 (ham correctly detected)

**Precision**
$$
\text{Precision} = \frac{70}{70 + 30} = 0.70
$$

**Recall**
$$
\text{Recall} = \frac{70}{70 + 10} = 0.875
$$

**F1-score**
$$
F1 = \frac{2 \cdot (0.70 \cdot 0.875)}{0.70 + 0.875} \approx 0.778
$$



In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = [1,1,1,1,0,0,0,0,0,0]  # actual labels (1=spam, 0=ham)
y_pred = [1,1,0,1,0,1,0,0,0,0]  # predicted labels

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)


Precision: 0.75
Recall: 0.75
F1-score: 0.75


# Avoiding Harms in Classification

## 1. Sources of Harm
- **Allocative harm**: wrong or unequal access to opportunities/resources (e.g., loans, jobs).
- **Representational harm**: stereotypes or erasure (e.g., mislabeling names, accents).
- **Procedural harm**: lack of due process, transparency, or recourse.

These harms often arise from (i) biased data \( \mathcal{D} \), (ii) misaligned objective functions, (iii) deployment/context mismatch.

## 2. Setup and Notation
Let inputs \(X \in \mathbb{R}^d\), label \(Y \in \{0,1\}\), sensitive attribute \(A \in \mathcal{A}\) (e.g., gender, race).
A classifier outputs a score \( s_\theta(X) = P_\theta(Y=1\mid X) \) and prediction
\[
\hat{Y}_t = \mathbb{I}\{ s_\theta(X) \ge t \}.
\]

## 3. Key Fairness Metrics
Let \( \mathbb{P}_a(\cdot) := \mathbb{P}(\cdot \mid A=a) \).

1. **Demographic Parity (DP)**
\[
\text{DP gap} := \left| \mathbb{P}_{a}(\hat{Y}=1) - \mathbb{P}_{b}(\hat{Y}=1) \right|.
\]
\[
\text{Disparate Impact} := \frac{\mathbb{P}_{a}(\hat{Y}=1)}{\mathbb{P}_{b}(\hat{Y}=1)}.
\]

2. **Equal Opportunity (EOpp)** — equal True Positive Rates:
\[
\text{EOpp gap} := \left| \mathbb{P}_{a}(\hat{Y}=1 \mid Y=1) - \mathbb{P}_{b}(\hat{Y}=1 \mid Y=1) \right|.
\]

3. **Equalized Odds (EOdds)** — equal TPR and FPR:
\[
\text{EOdds gap} := \max\Big(
\left| \mathbb{P}_{a}(\hat{Y}=1 \mid Y=1) - \mathbb{P}_{b}(\hat{Y}=1 \mid Y=1) \right|,
\left| \mathbb{P}_{a}(\hat{Y}=1 \mid Y=0) - \mathbb{P}_{b}(\hat{Y}=1 \mid Y=0) \right|
\Big).
\]

4. **Predictive Parity (PPV parity)** — equal precision:
\[
\text{PPV gap} := \left| \mathbb{P}_{a}(Y=1 \mid \hat{Y}=1) - \mathbb{P}_{b}(Y=1 \mid \hat{Y}=1) \right|.
\]

5. **Calibration within groups**:
A classifier is calibrated if for any score \(s\),
\[
\mathbb{P}(Y=1 \mid s_\theta(X)=s, A=a) \approx s \quad \forall a.
\]
(Practically assessed with groupwise Brier score or reliability curves.)

**Note:** Many metrics are mutually incompatible under base-rate differences; trade-offs are inevitable.

## 4. Risk, Costs, and Thresholds
With asymmetric costs \(c_{FN}, c_{FP}\), the Bayes-optimal global threshold is
\[
t^\star = \frac{c_{FP}}{c_{FP} + c_{FN}}.
\]
To reduce group disparities, use **group-specific thresholds** \( t_a \) selected to meet EOpp/EOdds while controlling utility.

## 5. Mitigation Strategies
- **Pre-processing**:
  - De-bias sampling or **reweighing**: assign weights \(w(X,A,Y)\) so empirical distributions align across groups.
  - Repair features (learn transformations that reduce correlations with \(A\)).
- **In-processing**:
  - Constrained ERM:
    \[
    \min_\theta \ \mathcal{L}(\theta) \quad \text{s.t.} \quad g_k(\theta) \le \epsilon
    \]
    where \(g_k\) encode fairness constraints (e.g., EOpp gap).
  - Penalty form:
    \[
    \min_\theta \ \mathcal{L}(\theta) + \lambda \cdot \Phi(\theta),
    \]
    with \(\Phi\) measuring DP/EOpp/EOdds violations.
  - Adversarial debiasing: learn \(s_\theta\) while an adversary predicts \(A\) from representations; minimize task loss, maximize adversary loss.
- **Post-processing**:
  - **Threshold adjustment** (per-group \(t_a\)) to satisfy EOpp/EOdds.
  - **Randomized post-processing** (Hardt et al., 2016): mix decisions at the score boundary to meet constraints.

## 6. Uncertainty and Abstention
When scores are low-confidence, use **selective classification** (reject-option):
\[
\hat{Y} =
\begin{cases}
1 & s_\theta(X) \ge t_1\\
0 & s_\theta(X) \le t_0\\
\text{abstain} & \text{otherwise}
\end{cases}
\]
Abstention can reduce harms by deferring to human review.

## 7. Governance and Deployment
- Data documentation (datasheets, model cards), stakeholder review, recourse procedures.
- Ongoing monitoring: track metrics over time and under distribution shift.
- Privacy (e.g., differential privacy) and security reduce downstream harms.


In [10]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ---- 1) Create a toy dataset with bias (for demo) ----
rng = np.random.default_rng(7)
n = 4000

# Sensitive attribute A in {0,1}
A = rng.integers(0, 2, size=n)

# Features: X1 informative but distribution differs by A (induces base-rate differences)
X1 = rng.normal(loc=0.0 + 0.8*A, scale=1.0, size=n)
X2 = rng.normal(loc=0.0, scale=1.0, size=n)
X3 = rng.normal(loc=0.0, scale=1.0, size=n)

X = np.c_[X1, X2, X3]
# Ground-truth probability with different base rates per group
logit = 0.6*X1 + 0.3*X2 - 0.2*X3 - 0.3 + 0.5*A  # higher base rate for A=1
p = 1 / (1 + np.exp(-logit))
Y = rng.binomial(1, p)

df = pd.DataFrame(X, columns=["x1","x2","x3"])
df["A"] = A
df["y"] = Y

# Train/test split (preserve distribution)
X_tr, X_te, y_tr, y_te, A_tr, A_te = train_test_split(
    df[["x1","x2","x3"]].values, df["y"].values, df["A"].values, test_size=0.35, random_state=42, stratify=df["y"].values
)

# ---- 2) Baseline model ----
scaler = StandardScaler()
X_trs = scaler.fit_transform(X_tr)
X_tes = scaler.transform(X_te)

clf = LogisticRegression(max_iter=200)
clf.fit(X_trs, y_tr)

scores = clf.predict_proba(X_tes)[:,1]

# ---- 3) Helper metrics by group ----
def group_metrics(scores, y, A, thr=0.5):
    yhat = (scores >= thr).astype(int)

    def rate(mask, cond=None):
        if cond is None:
            return yhat[mask].mean()
        return ((yhat==1) & cond)[mask].sum() / max(1, cond[mask].sum())

    metrics = {}
    for a in [0,1]:
        m = (A==a)
        # Positive rate
        pr = yhat[m].mean()
        # TPR/FPR
        tpr = ((yhat==1) & (y==1))[m].sum() / max(1, (y==1)[m].sum())
        fpr = ((yhat==1) & (y==0))[m].sum() / max(1, (y==0)[m].sum())
        # PPV (precision)
        ppv = ((y==1) & (yhat==1))[m].sum() / max(1, (yhat==1)[m].sum())
        # Brier (calibration-ish)
        brier = brier_score_loss(y[m], scores[m])
        metrics[a] = dict(PR=pr, TPR=tpr, FPR=fpr, PPV=ppv, Brier=brier, N=m.sum())
    # Gaps
    dp_gap = abs(metrics[1]["PR"] - metrics[0]["PR"])
    eopp_gap = abs(metrics[1]["TPR"] - metrics[0]["TPR"])
    eodds_gap = max(abs(metrics[1]["TPR"] - metrics[0]["TPR"]), abs(metrics[1]["FPR"] - metrics[0]["FPR"]))
    ppv_gap = abs(metrics[1]["PPV"] - metrics[0]["PPV"])
    return metrics, dict(DP_gap=dp_gap, EOpp_gap=eopp_gap, EOdds_gap=eodds_gap, PPV_gap=ppv_gap)

base_m, base_gaps = group_metrics(scores, y_te, A_te, thr=0.5)

print("=== Baseline (global threshold 0.5) ===")
print("AUC:", roc_auc_score(y_te, scores))
print("Group 0:", base_m[0])
print("Group 1:", base_m[1])
print("Gaps:", base_gaps, "\n")

# ---- 4) Simple pre-processing: Reweighing (inverse propensity by A,Y) ----
# Estimate weights so P(Y|A) becomes closer across groups
df_tr = pd.DataFrame(X_tr, columns=["x1","x2","x3"])
df_tr["A"] = A_tr
df_tr["y"] = y_tr

# empirical P(A=a, Y=y)
joint = df_tr.groupby(["A","y"]).size() / len(df_tr)
# target: factorized P(A)*P(Y) (independence) -> product of marginals
pA = df_tr["A"].value_counts(normalize=True)
pY = df_tr["y"].value_counts(normalize=True)
target = {(a,y): pA[a]*pY[y] for a in [0,1] for y in [0,1]}

# weights w = target / empirical
def weight_row(a,y):
    return target[(a,y)] / max(1e-9, joint[(a,y)])

w_tr = np.array([weight_row(a,y) for a,y in zip(A_tr, y_tr)])

# Refit with sample weights
clf_rw = LogisticRegression(max_iter=200)
clf_rw.fit(scaler.fit_transform(X_tr), y_tr, sample_weight=w_tr)

scores_rw = clf_rw.predict_proba(scaler.transform(X_te))[:,1]
rw_m, rw_gaps = group_metrics(scores_rw, y_te, A_te, thr=0.5)

print("=== After Reweighing (still global thr=0.5) ===")
print("AUC:", roc_auc_score(y_te, scores_rw))
print("Group 0:", rw_m[0])
print("Group 1:", rw_m[1])
print("Gaps:", rw_gaps, "\n")

# ---- 5) Post-processing: pick per-group thresholds for Equal Opportunity ----
# Grid-search thresholds t0,t1 to minimize |TPR gap| subject to small utility loss
ths = np.linspace(0.2, 0.8, 25)

def tpr(scores, y, mask, thr):
    yhat = (scores>=thr).astype(int)
    den = max(1, (y[mask]==1).sum())
    return ((yhat==1) & (y==1) & mask).sum() / den

def util(scores, y, thr):
    # simple utility: accuracy
    yhat = (scores>=thr).astype(int)
    return (yhat==y).mean()

best = None
best_pair = None
for t0 in ths:
    for t1 in ths:
        m0 = (A_te==0); m1 = (A_te==1)
        tpr0 = tpr(scores_rw, y_te, m0, t0)
        tpr1 = tpr(scores_rw, y_te, m1, t1)
        gap = abs(tpr0 - tpr1)
        # utility as average of per-group accuracies
        yhat = np.where(A_te==0, (scores_rw>=t0).astype(int), (scores_rw>=t1).astype(int))
        acc = (yhat==y_te).mean()
        # track Pareto-ish: minimize gap, break ties by higher acc
        key = (gap, -acc)
        if (best is None) or (key < best):
            best = key
            best_pair = (t0, t1, gap, acc)

t0_opt, t1_opt, gap_opt, acc_opt = best_pair
# Metrics with group thresholds
yhat_pp = np.where(A_te==0, (scores_rw>=t0_opt).astype(int), (scores_rw>=t1_opt).astype(int))
def group_metrics_from_preds(yhat, y, A):
    out = {}
    for a in [0,1]:
        m = (A==a)
        pr = yhat[m].mean()
        tpr = ((yhat==1) & (y==1))[m].sum() / max(1,(y==1)[m].sum())
        fpr = ((yhat==1) & (y==0))[m].sum() / max(1,(y==0)[m].sum())
        ppv = ((y==1) & (yhat==1))[m].sum() / max(1,(yhat==1)[m].sum())
        out[a] = dict(PR=pr, TPR=tpr, FPR=fpr, PPV=ppv, N=m.sum())
    gaps = dict(
        DP_gap=abs(out[1]["PR"]-out[0]["PR"]),
        EOpp_gap=abs(out[1]["TPR"]-out[0]["TPR"]),
        EOdds_gap=max(abs(out[1]["TPR"]-out[0]["TPR"]), abs(out[1]["FPR"]-out[0]["FPR"])),
        PPV_gap=abs(out[1]["PPV"]-out[0]["PPV"])
    )
    return out, gaps

pp_m, pp_gaps = group_metrics_from_preds(yhat_pp, y_te, A_te)

print("=== Post-processing: group thresholds for Equal Opportunity ===")
print(f"Chosen thresholds: t0={t0_opt:.3f}, t1={t1_opt:.3f}")
print("Accuracy (with group thresholds):", ((yhat_pp==y_te).mean()))
print("Group 0:", pp_m[0])
print("Group 1:", pp_m[1])
print("Gaps:", pp_gaps)


=== Baseline (global threshold 0.5) ===
AUC: 0.6864495450785774
Group 0: {'PR': np.float64(0.5172910662824207), 'TPR': np.float64(0.6561514195583596), 'FPR': np.float64(0.4005305039787798), 'PPV': np.float64(0.5793871866295265), 'Brier': 0.2302815984288431, 'N': np.int64(694)}
Group 1: {'PR': np.float64(0.7549575070821529), 'TPR': np.float64(0.8207343412526998), 'FPR': np.float64(0.6296296296296297), 'PPV': np.float64(0.7129455909943715), 'Brier': 0.21140219630414195, 'N': np.int64(706)}
Gaps: {'DP_gap': np.float64(0.2376664407997322), 'EOpp_gap': np.float64(0.1645829216943402), 'EOdds_gap': np.float64(0.22909912565084983), 'PPV_gap': np.float64(0.133558404364845)} 

=== After Reweighing (still global thr=0.5) ===
AUC: 0.6816149710504549
Group 0: {'PR': np.float64(0.5547550432276657), 'TPR': np.float64(0.6908517350157729), 'FPR': np.float64(0.4403183023872679), 'PPV': np.float64(0.5688311688311688), 'Brier': 0.23208890603614213, 'N': np.int64(694)}
Group 1: {'PR': np.float64(0.77195467